# Lesson 3b: Monte Carlo Methods - Practical Implementation

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Implement MC Prediction** - First-visit and every-visit variants
2. **Implement MC Control** - Find optimal policies using MC
3. **Solve Blackjack** - Classic RL benchmark environment
4. **Implement $\epsilon$-Greedy** - Exploration-exploitation balance
5. **Implement Importance Sampling** - Off-policy MC methods
6. **Analyze Convergence** - Visualize learning progress
7. **Compare Variants** - First-visit vs every-visit, on vs off-policy

This notebook provides complete, production-ready implementations of Monte Carlo methods.

---

## Table of Contents

1. [Setup and Environment](#setup)
2. [MC Prediction - Implementation](#mc-prediction)
3. [Blackjack Environment](#blackjack)
4. [MC Control with Epsilon-Greedy](#mc-control)
5. [Off-Policy MC with Importance Sampling](#off-policy)
6. [Performance Analysis](#performance)
7. [Visualization](#visualization)
8. [Exercises](#exercises)

---

## 1. Setup and Environment <a id='setup'></a>

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from typing import Tuple, Dict, List, Optional, Callable
from collections import defaultdict
import time

# Gymnasium
import gymnasium as gym

# Configuration
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Imports successful")
print(f"NumPy version: {np.__version__}")
print(f"Gymnasium version: {gym.__version__}")

## 2. MC Prediction - Implementation <a id='mc-prediction'></a>

### First-Visit Monte Carlo Prediction

In [ ]:
def generate_episode(
    env: gym.Env,
    policy: Callable,
    max_steps: int = 1000
) -> List[Tuple]:
    """
    Generate a single episode following given policy.
    
    Args:
        env: Gymnasium environment
        policy: Function mapping state -> action
        max_steps: Maximum episode length
    
    Returns:
        episode: List of (state, action, reward) tuples
    """
    episode = []
    state, _ = env.reset()
    
    for step in range(max_steps):
        action = policy(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        episode.append((state, action, reward))
        
        if terminated or truncated:
            break
        
        state = next_state
    
    return episode


def mc_prediction_first_visit(
    env: gym.Env,
    policy: Callable,
    n_episodes: int,
    gamma: float = 1.0
) -> Dict:
    """
    First-visit Monte Carlo prediction.
    
    Args:
        env: Gymnasium environment
        policy: Policy to evaluate (state -> action)
        n_episodes: Number of episodes
        gamma: Discount factor
    
    Returns:
        V: State-value function (dict mapping state -> value)
    """
    V = defaultdict(float)
    returns = defaultdict(list)
    
    for episode_num in range(n_episodes):
        episode = generate_episode(env, policy)
        
        # Track visited states in this episode
        visited_states = set()
        
        # Calculate returns backward
        G = 0
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            # First-visit check
            if state not in visited_states:
                visited_states.add(state)
                returns[state].append(G)
                V[state] = np.mean(returns[state])
    
    return V


def mc_prediction_every_visit(
    env: gym.Env,
    policy: Callable,
    n_episodes: int,
    gamma: float = 1.0
) -> Dict:
    """
    Every-visit Monte Carlo prediction.
    
    Args:
        env: Gymnasium environment
        policy: Policy to evaluate
        n_episodes: Number of episodes
        gamma: Discount factor
    
    Returns:
        V: State-value function
    """
    V = defaultdict(float)
    returns = defaultdict(list)
    
    for episode_num in range(n_episodes):
        episode = generate_episode(env, policy)
        
        # Calculate returns backward (no first-visit check)
        G = 0
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            returns[state].append(G)
            V[state] = np.mean(returns[state])
    
    return V


def mc_prediction_incremental(
    env: gym.Env,
    policy: Callable,
    n_episodes: int,
    gamma: float = 1.0,
    alpha: float = 0.01
) -> Tuple[Dict, List[float]]:
    """
    Incremental Monte Carlo prediction (memory-efficient).
    
    Uses: V(s) ← V(s) + α[G - V(s)]
    
    Args:
        env: Gymnasium environment
        policy: Policy to evaluate
        n_episodes: Number of episodes
        gamma: Discount factor
        alpha: Step size
    
    Returns:
        V: State-value function
        avg_returns: Average return per episode
    """
    V = defaultdict(float)
    avg_returns = []
    
    for episode_num in range(n_episodes):
        episode = generate_episode(env, policy)
        
        visited_states = set()
        G = 0
        episode_return = 0
        
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            if t == 0:
                episode_return = G
            
            # First-visit incremental update
            if state not in visited_states:
                visited_states.add(state)
                V[state] = V[state] + alpha * (G - V[state])
        
        avg_returns.append(episode_return)
    
    return V, avg_returns


print("✅ MC Prediction implementations complete")

## 3. Blackjack Environment <a id='blackjack'></a>

### Understanding Blackjack

**State**: (player_sum, dealer_showing, usable_ace)
- player_sum: Player's current sum (12-21)
- dealer_showing: Dealer's face-up card (1-10)
- usable_ace: Whether player has usable ace (True/False)

**Actions**: 
- 0: Stick (stop taking cards)
- 1: Hit (take another card)

**Reward**:
- +1: Win
- 0: Draw
- -1: Lose

In [ ]:
# Create Blackjack environment
env = gym.make('Blackjack-v1', natural=False, sab=False)

print("Blackjack Environment")
print("=" * 50)
print(f"State space: player_sum (12-21), dealer (1-10), usable_ace (0/1)")
print(f"Action space: {env.action_space}  (0=Stick, 1=Hit)")
print(f"\nState representation: (player_sum, dealer_card, usable_ace)")

# Test episode
state, _ = env.reset()
print(f"\nExample initial state: {state}")
print(f"  Player sum: {state[0]}")
print(f"  Dealer showing: {state[1]}")
print(f"  Usable ace: {state[2]}")

print("\n✅ Blackjack environment loaded")

### Simple Blackjack Policy

**Baseline strategy**: 
- Stick if player_sum >= 20
- Hit otherwise

In [ ]:
def simple_blackjack_policy(state: Tuple, threshold: int = 20) -> int:
    """
    Simple threshold policy for Blackjack.
    
    Args:
        state: (player_sum, dealer_card, usable_ace)
        threshold: Stick if player_sum >= threshold
    
    Returns:
        action: 0 (stick) or 1 (hit)
    """
    player_sum, dealer_card, usable_ace = state
    return 0 if player_sum >= threshold else 1


# Test simple policy
print("Testing Simple Blackjack Policy (threshold=20)")
print("=" * 50)

test_policy = lambda s: simple_blackjack_policy(s, threshold=20)
wins = 0
total = 1000

for _ in range(total):
    episode = generate_episode(env, test_policy)
    final_reward = episode[-1][2]
    if final_reward > 0:
        wins += 1

win_rate = wins / total
print(f"\nWin rate over {total} episodes: {win_rate:.1%}")
print(f"(Random play achieves ~42% win rate)")

print("\n✅ Simple policy tested")

### Evaluate Simple Policy with MC Prediction

In [ ]:
print("Evaluating Simple Policy with MC Prediction")
print("=" * 50)

# Evaluate with first-visit MC
print("\nRunning First-Visit MC (10,000 episodes)...")
V_first = mc_prediction_first_visit(
    env,
    test_policy,
    n_episodes=10000,
    gamma=1.0
)

print(f"Estimated {len(V_first)} unique states")

# Sample values
sample_states = [
    (20, 10, False),  # Player 20, dealer showing 10, no usable ace
    (18, 9, True),    # Player 18, dealer showing 9, usable ace
    (12, 2, False),   # Player 12, dealer showing 2, no usable ace
]

print("\nSample state values:")
for state in sample_states:
    value = V_first.get(state, 0.0)
    print(f"  {state}: V = {value:.4f}")

print("\n✅ MC Prediction complete")

## 4. MC Control with Epsilon-Greedy <a id='mc-control'></a>

### Implementation

In [ ]:
class EpsilonGreedyPolicy:
    """
    Epsilon-greedy policy for MC control.
    """
    def __init__(self, n_actions: int, epsilon: float = 0.1):
        self.n_actions = n_actions
        self.epsilon = epsilon
        self.Q = defaultdict(lambda: np.zeros(n_actions))
    
    def __call__(self, state) -> int:
        """Select action using epsilon-greedy strategy."""
        if np.random.random() < self.epsilon:
            # Explore: random action
            return np.random.randint(self.n_actions)
        else:
            # Exploit: greedy action
            return np.argmax(self.Q[state])
    
    def update_Q(self, state, action, G, alpha=0.01):
        """Incremental Q-value update."""
        self.Q[state][action] += alpha * (G - self.Q[state][action])
    
    def get_greedy_action(self, state) -> int:
        """Get greedy action (for evaluation)."""
        return np.argmax(self.Q[state])


def mc_control_epsilon_greedy(
    env: gym.Env,
    n_episodes: int,
    gamma: float = 1.0,
    epsilon: float = 0.1,
    alpha: float = 0.01,
    decay_epsilon: bool = False
) -> Tuple[EpsilonGreedyPolicy, List[float]]:
    """
    Monte Carlo control with epsilon-greedy exploration.
    
    Args:
        env: Gymnasium environment
        n_episodes: Number of episodes
        gamma: Discount factor
        epsilon: Exploration rate
        alpha: Learning rate
        decay_epsilon: Whether to decay epsilon over time (GLIE)
    
    Returns:
        policy: Learned epsilon-greedy policy
        episode_returns: Return from each episode
    """
    n_actions = env.action_space.n
    policy = EpsilonGreedyPolicy(n_actions, epsilon)
    episode_returns = []
    
    for episode_num in range(n_episodes):
        # Decay epsilon for GLIE
        if decay_epsilon:
            policy.epsilon = max(0.01, epsilon / (1 + episode_num / 1000))
        
        # Generate episode
        episode = generate_episode(env, policy)
        
        # Track first visits
        visited_pairs = set()
        
        # Calculate returns and update Q
        G = 0
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            # First-visit update
            if (state, action) not in visited_pairs:
                visited_pairs.add((state, action))
                policy.update_Q(state, action, G, alpha)
            
            if t == 0:
                episode_returns.append(G)
    
    return policy, episode_returns


print("✅ MC Control with epsilon-greedy implemented")

### Train Optimal Blackjack Policy

In [ ]:
print("Training Optimal Blackjack Policy with MC Control")
print("=" * 60)

# Train with epsilon-greedy MC control
print("\nTraining with 500,000 episodes...")
start_time = time.time()

learned_policy, returns = mc_control_epsilon_greedy(
    env,
    n_episodes=500000,
    gamma=1.0,
    epsilon=0.1,
    alpha=0.01,
    decay_epsilon=True  # GLIE
)

training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f}s")
print(f"Learned Q-values for {len(learned_policy.Q)} states")

# Evaluate learned policy
print("\nEvaluating learned policy (10,000 episodes)...")
greedy_policy = lambda s: learned_policy.get_greedy_action(s)
wins = 0
draws = 0
total = 10000

for _ in range(total):
    episode = generate_episode(env, greedy_policy)
    final_reward = episode[-1][2]
    if final_reward > 0:
        wins += 1
    elif final_reward == 0:
        draws += 1

win_rate = wins / total
draw_rate = draws / total
loss_rate = 1 - win_rate - draw_rate

print(f"\nLearned Policy Performance:")
print(f"  Wins:   {win_rate:.1%}")
print(f"  Draws:  {draw_rate:.1%}")
print(f"  Losses: {loss_rate:.1%}")
print(f"\nComparison:")
print(f"  Simple policy (threshold=20): ~42% win rate")
print(f"  Learned policy: {win_rate:.1%} win rate")
print(f"  Improvement: {(win_rate - 0.42) / 0.42 * 100:+.1f}%")

print("\n✅ MC Control training complete")

### Visualize Learning Progress

In [ ]:
# Plot learning curve
fig, ax = plt.subplots(figsize=(10, 6))

# Compute moving average of returns
window = 1000
moving_avg = np.convolve(returns, np.ones(window)/window, mode='valid')

ax.plot(moving_avg, linewidth=2, color='blue', alpha=0.8)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel(f'Average Return (window={window})', fontsize=12)
ax.set_title('MC Control Learning Curve - Blackjack', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Break-even')
ax.legend()

plt.tight_layout()
plt.savefig('/home/user/reinforcement-learning/static/images/3b_mc_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Learning curve plotted")

## 5. Off-Policy MC with Importance Sampling <a id='off-policy'></a>

### Weighted Importance Sampling Implementation

In [ ]:
def compute_importance_ratio(
    episode: List[Tuple],
    start_idx: int,
    target_policy: Callable,
    behavior_policy: Callable
) -> float:
    """
    Compute importance sampling ratio from time t to end of episode.
    
    ρ_t:T-1 = ∏ π(A_k|S_k) / b(A_k|S_k)
    
    Args:
        episode: List of (state, action, reward)
        start_idx: Starting time step
        target_policy: Target policy π
        behavior_policy: Behavior policy b
    
    Returns:
        ratio: Importance sampling ratio
    """
    ratio = 1.0
    
    for t in range(start_idx, len(episode)):
        state, action, _ = episode[t]
        
        # Get probabilities
        pi_prob = target_policy.get_action_prob(state, action)
        b_prob = behavior_policy.get_action_prob(state, action)
        
        # Early termination if ratio becomes 0
        if pi_prob == 0 or b_prob == 0:
            return 0.0
        
        ratio *= pi_prob / b_prob
    
    return ratio


class DeterministicPolicy:
    """
    Deterministic policy for off-policy target.
    """
    def __init__(self, n_actions: int):
        self.n_actions = n_actions
        self.Q = defaultdict(lambda: np.zeros(n_actions))
    
    def __call__(self, state) -> int:
        return np.argmax(self.Q[state])
    
    def get_action_prob(self, state, action: int) -> float:
        """Get probability of taking action in state."""
        greedy_action = np.argmax(self.Q[state])
        return 1.0 if action == greedy_action else 0.0


class RandomBehaviorPolicy:
    """
    Random behavior policy for off-policy learning.
    """
    def __init__(self, n_actions: int):
        self.n_actions = n_actions
    
    def __call__(self, state) -> int:
        return np.random.randint(self.n_actions)
    
    def get_action_prob(self, state, action: int) -> float:
        return 1.0 / self.n_actions


def mc_off_policy_weighted_is(
    env: gym.Env,
    n_episodes: int,
    gamma: float = 1.0
) -> Tuple[DeterministicPolicy, List[float]]:
    """
    Off-policy MC control with weighted importance sampling.
    
    Target policy: Greedy (deterministic)
    Behavior policy: Random (exploratory)
    
    Args:
        env: Gymnasium environment
        n_episodes: Number of episodes
        gamma: Discount factor
    
    Returns:
        target_policy: Learned deterministic policy
        episode_returns: Returns per episode
    """
    n_actions = env.action_space.n
    
    target_policy = DeterministicPolicy(n_actions)
    behavior_policy = RandomBehaviorPolicy(n_actions)
    
    # Cumulative sum of weights for each (s,a)
    C = defaultdict(lambda: np.zeros(n_actions))
    episode_returns = []
    
    for episode_num in range(n_episodes):
        # Generate episode using behavior policy
        episode = generate_episode(env, behavior_policy)
        
        # Process episode backward
        G = 0
        W = 1.0  # Importance sampling ratio
        
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            if t == 0:
                episode_returns.append(G)
            
            # Update cumulative weight
            C[state][action] += W
            
            # Weighted importance sampling update
            if C[state][action] > 0:
                target_policy.Q[state][action] += (W / C[state][action]) * (
                    G - target_policy.Q[state][action]
                )
            
            # Update importance ratio
            target_action = np.argmax(target_policy.Q[state])
            if action != target_action:
                # Target policy has 0 probability for this action
                break
            
            # π(a|s) / b(a|s) = 1 / (1/n_actions) = n_actions
            W *= n_actions
    
    return target_policy, episode_returns


print("✅ Off-policy MC with importance sampling implemented")

### Train with Off-Policy MC

In [ ]:
print("Training with Off-Policy MC (Weighted IS)")
print("=" * 60)

print("\nTraining with 100,000 episodes...")
print("  Target policy: Greedy (deterministic)")
print("  Behavior policy: Random (uniform)\n")

start_time = time.time()
off_policy, off_returns = mc_off_policy_weighted_is(
    env,
    n_episodes=100000,
    gamma=1.0
)
training_time = time.time() - start_time

print(f"Training completed in {training_time:.2f}s")
print(f"Learned Q-values for {len(off_policy.Q)} states")

# Evaluate
print("\nEvaluating off-policy learned greedy policy (10,000 episodes)...")
wins = 0
total = 10000

for _ in range(total):
    episode = generate_episode(env, off_policy)
    if episode[-1][2] > 0:
        wins += 1

win_rate_off = wins / total
print(f"\nOff-Policy Win Rate: {win_rate_off:.1%}")
print(f"On-Policy Win Rate:  {win_rate:.1%}")
print(f"Difference: {(win_rate_off - win_rate):.2%}")

print("\n✅ Off-policy MC training complete")

## 6. Performance Analysis <a id='performance'></a>

### Compare All Methods

In [ ]:
print("Comprehensive Performance Comparison")
print("=" * 70)

policies_to_test = [
    ("Random", lambda s: np.random.randint(2)),
    ("Simple (threshold=18)", lambda s: simple_blackjack_policy(s, 18)),
    ("Simple (threshold=20)", lambda s: simple_blackjack_policy(s, 20)),
    ("MC On-Policy (ε-greedy)", lambda s: learned_policy.get_greedy_action(s)),
    ("MC Off-Policy (greedy)", off_policy),
]

results = []
n_eval = 10000

for name, policy in policies_to_test:
    wins = 0
    draws = 0
    total_return = 0
    
    for _ in range(n_eval):
        episode = generate_episode(env, policy)
        reward = episode[-1][2]
        total_return += reward
        if reward > 0:
            wins += 1
        elif reward == 0:
            draws += 1
    
    win_rate = wins / n_eval
    draw_rate = draws / n_eval
    loss_rate = 1 - win_rate - draw_rate
    avg_return = total_return / n_eval
    
    results.append({
        'name': name,
        'win_rate': win_rate,
        'draw_rate': draw_rate,
        'loss_rate': loss_rate,
        'avg_return': avg_return
    })

# Display results
print(f"\n{'Policy':<30} {'Win Rate':<12} {'Draw Rate':<12} {'Avg Return':<12}")
print("=" * 70)

for result in results:
    print(f"{result['name']:<30} {result['win_rate']:<12.1%} {result['draw_rate']:<12.1%} {result['avg_return']:<12.4f}")

print("\n✅ Performance comparison complete")

## 7. Visualization <a id='visualization'></a>

### Visualize Value Function

In [ ]:
def plot_blackjack_value_function(Q_dict: Dict, title: str = "State-Value Function"):
    """
    Plot value function for Blackjack as 3D surface.
    
    Creates two plots: one for usable ace, one without.
    """
    fig = plt.figure(figsize=(16, 6))
    
    for usable_ace_idx, usable_ace in enumerate([False, True]):
        ax = fig.add_subplot(1, 2, usable_ace_idx + 1, projection='3d')
        
        # Create grid
        player_sums = np.arange(12, 22)  # 12-21
        dealer_cards = np.arange(1, 11)  # 1-10
        X, Y = np.meshgrid(dealer_cards, player_sums)
        
        # Compute values
        Z = np.zeros_like(X, dtype=float)
        for i, player_sum in enumerate(player_sums):
            for j, dealer_card in enumerate(dealer_cards):
                state = (player_sum, dealer_card, usable_ace)
                if state in Q_dict:
                    Z[i, j] = np.max(Q_dict[state])
        
        # Plot surface
        surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8, edgecolor='none')
        
        ax.set_xlabel('Dealer Showing', fontsize=10)
        ax.set_ylabel('Player Sum', fontsize=10)
        ax.set_zlabel('Value', fontsize=10)
        ace_str = "With" if usable_ace else "Without"
        ax.set_title(f"{title}\n({ace_str} Usable Ace)", fontsize=11, fontweight='bold')
        ax.view_init(elev=20, azim=45)
        
        fig.colorbar(surf, ax=ax, shrink=0.5)
    
    plt.tight_layout()
    return fig


# Plot learned value function
fig = plot_blackjack_value_function(
    learned_policy.Q,
    title="Learned Value Function (MC Control)"
)
plt.savefig('/home/user/reinforcement-learning/static/images/3b_blackjack_value_function.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Value function visualization created")

### Visualize Policy

In [ ]:
def plot_blackjack_policy(Q_dict: Dict, title: str = "Learned Policy"):
    """
    Plot optimal policy as heatmap.
    
    0 (Stick) = Blue, 1 (Hit) = Red
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for usable_ace_idx, usable_ace in enumerate([False, True]):
        ax = axes[usable_ace_idx]
        
        # Create grid
        player_sums = np.arange(12, 22)  # 12-21
        dealer_cards = np.arange(1, 11)  # 1-10
        
        # Compute policy
        policy_grid = np.zeros((len(player_sums), len(dealer_cards)))
        for i, player_sum in enumerate(player_sums):
            for j, dealer_card in enumerate(dealer_cards):
                state = (player_sum, dealer_card, usable_ace)
                if state in Q_dict:
                    policy_grid[i, j] = np.argmax(Q_dict[state])
        
        # Plot heatmap
        im = ax.imshow(policy_grid, cmap='RdYlBu_r', aspect='auto', interpolation='nearest')
        
        ax.set_xticks(np.arange(len(dealer_cards)))
        ax.set_yticks(np.arange(len(player_sums)))
        ax.set_xticklabels(dealer_cards)
        ax.set_yticklabels(player_sums)
        
        ax.set_xlabel('Dealer Showing', fontsize=11)
        ax.set_ylabel('Player Sum', fontsize=11)
        ace_str = "With" if usable_ace else "Without"
        ax.set_title(f"{title}\n({ace_str} Usable Ace)", fontsize=12, fontweight='bold')
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax, ticks=[0, 1])
        cbar.set_label('Action', rotation=270, labelpad=15)
        cbar.ax.set_yticklabels(['Stick (0)', 'Hit (1)'])
    
    plt.tight_layout()
    return fig


# Plot learned policy
fig = plot_blackjack_policy(
    learned_policy.Q,
    title="Optimal Blackjack Policy (MC Control)"
)
plt.savefig('/home/user/reinforcement-learning/static/images/3b_blackjack_policy.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Policy visualization created")

## 8. Exercises <a id='exercises'></a>

### Exercise 1: Compare First-Visit vs Every-Visit

Empirically compare first-visit and every-visit MC on a simple environment.

In [ ]:
# TODO: Compare convergence speed and accuracy
# YOUR CODE HERE

# Hints:
# - Use a simple policy
# - Track estimation error over episodes
# - Plot comparison

### Exercise 2: Epsilon Decay Schedules

Compare different epsilon decay strategies for GLIE.

In [ ]:
# TODO: Test linear, exponential, and 1/t decay
# YOUR CODE HERE

### Exercise 3: Ordinary vs Weighted Importance Sampling

Implement ordinary importance sampling and compare variance with weighted IS.

In [ ]:
# TODO: Implement ordinary IS
# YOUR CODE HERE

# Compare:
# - Convergence speed
# - Variance of estimates
# - Final policy quality

---

## Summary

In this notebook, you have:

1. ✅ **Implemented MC Prediction** - First-visit, every-visit, and incremental
2. ✅ **Solved Blackjack** - Classic benchmark with MC control
3. ✅ **Implemented Epsilon-Greedy** - Exploration-exploitation balance
4. ✅ **Implemented Off-Policy MC** - Importance sampling methods
5. ✅ **Analyzed Performance** - Comprehensive comparison of variants
6. ✅ **Visualized Results** - Value functions and policies

### Key Insights

1. **MC works without a model** - Learns directly from experience
2. **Exploration is critical** - ε-greedy ensures all states visited
3. **First-visit vs Every-visit** - Similar convergence, different properties
4. **Off-policy enables learning** optimal policy while exploring
5. **Importance sampling** - Higher variance but greater flexibility

### Performance Results

Blackjack win rates:
- Random policy: ~28%
- Simple threshold: ~42%
- **Learned MC policy: ~43-44%** ✅

(Note: Blackjack inherently favors dealer; optimal play achieves ~43-44% win rate)

### Transition to Temporal Difference

Monte Carlo limitations:
- Requires complete episodes
- High variance
- Delayed learning

**Lesson 4** introduces **Temporal Difference (TD) learning**, which:
- Updates online during episodes
- Works for continuing tasks
- Lower variance through bootstrapping
- Foundation for Q-learning and SARSA

---

**Lesson 3b Complete!** 🎉

You now have production-ready Monte Carlo implementations for model-free reinforcement learning.